In [14]:
from data_frame.view.view_manager import ViewManager
from data_frame.spark_utils import get_spark

In [15]:
spark = get_spark(app_name="view manager")

In [16]:
# Create DataFrame
data = [
    (1, "Alice", 50000, "Engineering"),
    (2, "Bob", 60000, "Marketing"),
    (3, "Charlie", 55000, "Engineering"),
    (4, "David", 45000, "Sales")
]
df_emp = spark.createDataFrame(data, ["id", "name", "salary", "dept"])

## 1. Creating and Using Temp Views
"""

In [17]:
# Create temporary view
ViewManager.create_temp_view(df_emp, "employees")

# Query using SQL
result = spark.sql("""
    SELECT dept, 
           COUNT(*) as emp_count,
           AVG(salary) as avg_salary,
           MAX(salary) as max_salary
    FROM employees
    GROUP BY dept
    ORDER BY avg_salary DESC
""")
print("Department summary using SQL:")
result.show()

Department summary using SQL:
+-----------+---------+----------+----------+
|       dept|emp_count|avg_salary|max_salary|
+-----------+---------+----------+----------+
|  Marketing|        1|   60000.0|     60000|
|Engineering|        2|   52500.0|     55000|
|      Sales|        1|   45000.0|     45000|
+-----------+---------+----------+----------+



## 2. Global Views

In [18]:
# Create global view
global_view = ViewManager.create_global_view(df_emp, "global_employees")
print(f"Global view name: {global_view}")

# Access global view from another session (simulated)
# In a real scenario, this could be from a different SparkSession
spark.newSession().sql("""
    SELECT * FROM global_temp.global_employees 
    WHERE dept = 'Engineering'
""").show()

Global view name: global_temp.global_employees
+---+-------+------+-----------+
| id|   name|salary|       dept|
+---+-------+------+-----------+
|  1|  Alice| 50000|Engineering|
|  3|Charlie| 55000|Engineering|
+---+-------+------+-----------+



## 3. Managing Views

In [19]:
# List all views
print("All tables/views:")
tables = ViewManager.list_views(spark)
for table in tables:
    print(f"  - {table.name} ({table.tableType})")

All tables/views:
  - employees (TEMPORARY)


In [20]:
# Drop temp view
ViewManager.drop_view(spark, "employees")
print("\nAfter dropping 'employees':")
tables = ViewManager.list_views(spark)
for table in tables:
    print(f"  - {table.name}")


After dropping 'employees':
